# Phase 3 — TORGO Data Organization & Preprocessing

This notebook organizes the TORGO speech dataset into a clean, verified metadata catalog.

### Key Design Choices & Clinical Grounding:
1. **Microphone Channel Standardization**: Only `wav_arrayMic` recordings are indexed to ensure consistent room acoustics across all recordings and eliminate the microphone type confound.
2. **Explicit Whitelist Mapping**: Speaker diagnoses are mapped from an explicit dictionary rather than heuristic folder-name parsing.
3. **Clinical Severity Labels**: Grounded in Table 1 of Joy & Umesh (Interspeech 2017). Standardized into both a detailed clinical category (`moderate-severe` for M05) and a clean 3-class tier (`severity_3class`: `mild`, `moderate`, `severe`) for balanced downstream multi-class modeling.
4. **Duration & Quality Auditing**: Computes `duration_seconds` for all recordings using audio header inspection to flag corrupted or zero-length audio clips.

In [1]:
import os
import sys
from pathlib import Path
import pandas as pd
import soundfile as sf
from tqdm.notebook import tqdm

NOTEBOOK_DIR = Path.cwd().resolve()
if (NOTEBOOK_DIR / "data").is_dir():
    PROJECT_ROOT = NOTEBOOK_DIR
elif (NOTEBOOK_DIR.parent / "data").is_dir():
    PROJECT_ROOT = NOTEBOOK_DIR.parent
else:
    raise FileNotFoundError("Could not find the project's data directory.")

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.project_paths import DATA_DIR, METADATA_CSV, resolve_project_path
from src.data_loading import load_and_preprocess_audio, get_audio_info

# Explicit speaker whitelists grounded in clinical records
SPEAKER_LABELS = {
    "F01": "dysarthric", "F03": "dysarthric", "F04": "dysarthric",
    "M01": "dysarthric", "M02": "dysarthric", "M03": "dysarthric",
    "M04": "dysarthric", "M05": "dysarthric",
    "FC01": "control", "FC02": "control", "FC03": "control",
    "MC01": "control", "MC02": "control", "MC03": "control", "MC04": "control",
}

# Clinical severity labels reported in Joy & Umesh (Interspeech 2017), Table 1
SEVERITY_MAP = {
    "F01": "severe",
    "F03": "moderate",
    "F04": "mild",
    "M01": "severe",
    "M02": "severe",
    "M03": "mild",
    "M04": "severe",
    "M05": "moderate-severe"
}

# Standardized 3-tier severity mapping for clean multi-class evaluation
SEVERITY_3CLASS_MAP = {
    "F01": "severe",
    "F03": "moderate",
    "F04": "mild",
    "M01": "severe",
    "M02": "severe",
    "M03": "mild",
    "M04": "severe",
    "M05": "moderate"
}

def parse_torgo_directory(base_dir, project_root):
    """
    Scans the TORGO dataset directory structure and extracts metadata.
    Collects file paths, speaker ID, labels, clinical severity, and audio durations.
    """
    data = []
    print("Scanning TORGO array-microphone audio recordings...")
    for root, dirs, files in os.walk(base_dir):
        for file in files:
            if file.endswith('.wav'):
                if 'wav_arrayMic' not in root:
                    continue
                
                file_path = os.path.relpath(os.path.join(root, file), start=project_root)
                path_parts = root.split(os.sep)
                
                speaker_id = "UNKNOWN"
                for i, part in enumerate(path_parts):
                    if "Session" in part and i > 0:
                        speaker_id = path_parts[i-1]
                        break
                
                label = SPEAKER_LABELS.get(speaker_id)
                if label is None:
                    raise ValueError(f"Unexpected speaker ID '{speaker_id}' for {file_path}.")
                
                severity = "none" if label == "control" else SEVERITY_MAP.get(speaker_id, "unknown")
                severity_3class = "none" if label == "control" else SEVERITY_3CLASS_MAP.get(speaker_id, "unknown")
                
                # Fast audio header inspection
                try:
                    info = sf.info(resolve_project_path(file_path))
                    duration = round(float(info.duration), 3)
                except Exception:
                    duration = 0.0
                
                data.append({
                    "file_path": file_path,
                    "speaker_id": speaker_id,
                    "label": label,
                    "severity": severity,
                    "severity_3class": severity_3class,
                    "duration_seconds": duration,
                    "dataset_source": "TORGO"
                })
                
    return pd.DataFrame(data)

df = parse_torgo_directory(DATA_DIR, PROJECT_ROOT)
df.to_csv(METADATA_CSV, index=False)
print(f"Done! Indexed {len(df)} array microphone recordings.")
df.head()

Scanning TORGO array-microphone audio recordings...


Done! Indexed 9417 array microphone recordings.


,file_path,speaker_id,label,severity,severity_3class,duration_seconds,dataset_source
0,data\MC\F01\Session1\wav_arrayMic\0001.wav,F01,dysarthric,severe,severe,9.30,TORGO
1,data\MC\F01\Session1\wav_arrayMic\0002.wav,F01,dysarthric,severe,severe,8.40,TORGO
2,data\MC\F01\Session1\wav_arrayMic\0003.wav,F01,dysarthric,severe,severe,6.45,TORGO
3,data\MC\F01\Session1\wav_arrayMic\0004.wav,F01,dysarthric,severe,severe,8.70,TORGO
4,data\MC\F01\Session1\wav_arrayMic\0005.wav,F01,dysarthric,severe,severe,4.65,TORGO


In [2]:
print("=== Dataset Distribution Summary ===")
print(f"Total Audio Recordings: {len(df)}")
print(f"Total Unique Speakers:  {df['speaker_id'].nunique()}")

print("\n--- Breakdown by Diagnosis and Severity ---")
print(df.groupby(['label', 'severity']).size())

print("\n--- Standardized 3-Class Severity Breakdown ---")
print(df.groupby(['label', 'severity_3class']).size())

print("\n--- Per-Speaker Recording Counts ---")
print(df['speaker_id'].value_counts())

print("\n--- Audio Duration Statistics (Seconds) ---")
print(df['duration_seconds'].describe())

# Flag any empty or sub-second files
empty_files = df[df['duration_seconds'] == 0]
if len(empty_files) > 0:
    print(f"\n[WARNING] Found {len(empty_files)} zero-length audio file(s):")
    print(empty_files[['file_path', 'speaker_id', 'label']])

=== Dataset Distribution Summary ===
Total Audio Recordings: 9417
Total Unique Speakers:  15

--- Breakdown by Diagnosis and Severity ---
label       severity       
control     none               6236
dysarthric  mild                864
            moderate            848
            moderate-severe     128
            severe             1341
dtype: int64

--- Standardized 3-Class Severity Breakdown ---
label       severity_3class
control     none               6236
dysarthric  mild                864
            moderate            976
            severe             1341
dtype: int64

--- Per-Speaker Recording Counts ---
speaker_id
FC02    1261
MC01    1108
MC04    1021
FC03     995
MC03     900
F03      848
MC02     695
F04      448
M04      421
M03      416
M02      400
M01      386
FC01     256
F01      134
M05      128
Name: count, dtype: int64

--- Audio Duration Statistics (Seconds) ---
count    9417.000000
mean        3.087580
std         2.129274
min         0.000000
25%     

In [3]:
# Test audio loading with refined preprocessing (top_db=30, robust normalization)
test_file = df['file_path'].iloc[0]
y_norm, sr = load_and_preprocess_audio(test_file)

print(f"Success! Preprocessed waveform loaded with sample rate {sr} Hz.")
print(f"Shape: {y_norm.shape} | Peak amplitude: {y_norm.max():.4f} | Min amplitude: {y_norm.min():.4f}")

Success! Preprocessed waveform loaded with sample rate 16000 Hz.
Shape: (148800,) | Peak amplitude: 1.0000 | Min amplitude: -1.0000
